In [2]:
import mlx.nn as nn
import mlx.core as mx
import mlx.core.random as random
import mlx.optimizers as optim

In [3]:
# hyper_parameters
batch_size = 64
block_size = 256

eval_iters = 200
eval_interval = 500
max_iters = 1000

lr = 3e-4
n_emb = 256
n_head = 8
n_layer = 6
dropout = 0.2

In [62]:
with open("tiny_shakespeare.txt",'r') as f:
    text = f.read()

chars = sorted(list(set(text))) # we use set() as it eliminates duplicate characters
vocab_size = len(chars)

stoi = {ch:i for (i,ch) in enumerate(chars)}
itos = {i:ch for (ch,i) in stoi.items()}

encode = lambda txt: [stoi[c] for c in txt]
decode = lambda txt: ''.join([itos[i] for i in txt])

# encoding the entire dataset
data = mx.array(encode(text), dtype=mx.int64)

In [63]:
# splitting dataset into train and val
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [40]:
def get_batch(split):
    # generate a small batch of data of inputs x and outputs y
    data = train_data if split == 'train' else val_data

    ix = random.randint(0, len(data)-block_size, [batch_size])
    
    x = mx.stack([data[i.item():i.item()+block_size] for i in ix])
    y = mx.stack([data[i.item()+1:i.item()+block_size+1] for i in ix])
    return x, y

# first input to transformer
xb, yb = get_batch('train')

In [41]:
def estimate_loss(): # this averages loss over mutiple batches and its lot less noisy
    out = {}    
    for split in ['train','val']:
        losses = mx.zeros(eval_iters)
        
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits = model(X)
            losses[k] = nn.losses.cross_entropy(logits, Y, reduction="mean")
        
        out[split] = losses.mean()

    return out

In [42]:
# defining attention head
class Head(nn.Module):
    """ one head of self-attention """
    
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)
        self.tril = mx.tril(mx.ones((block_size, block_size), dtype=mx.float32))
        self.dropout = nn.Dropout(dropout)
    
    def __call__(self, x):
        B,T,C = x.shape
        k = self.key(x) # (B,T,C) where C -> head_size        
        q = self.query(x) # (B,T,C) where C -> head_size
        
        # compute attention scores ("affinities")
        wei = q @ k.transpose(0,2,1)  # (B,T,C) @ (B,C,T) -> (B,T,T)
        wei = wei / mx.sqrt(C)        
        wei = mx.where(self.tril[:T,:T]==1, wei, float('-inf')) # (B,T,T)        
        wei  = mx.softmax(wei, axis=-1) #(B,T,T)
        wei = self.dropout(wei)
        
        # perform weighted aggragation of the values
        v = self.value(x) # (B,T,C) where C -> head_size
        out = wei @ v # (B,T,T) @ (B,T,C) -> (B,T,C) where C -> head_size
        
        return out

In [43]:
# defining multihead-attention head
class MultiHeadAttention(nn.Module):
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads = [Head(head_size) for _ in range(num_heads)]
        self.proj = nn.Linear(n_emb, n_emb) # this is the projection layer going back into the residual pathway
        self.dropout = nn.Dropout(dropout)
    
    def __call__(self,x):
        # output of self-attentions
        out = mx.concatenate([h(x) for h in self.heads],axis=-1) # concatnating across channel dimension. we are feeding it into each self-attention heads        
        out = self.dropout(self.proj(out))
        return out

In [44]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity"""

    def __init__(self, n_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb, n_emb),
            nn.ReLU(), 
            nn.Linear(n_emb, n_emb), # this is the projection layer going back into the residual pathway
            nn.Dropout(dropout)
        )
    
    def __call__(self, x): # done on per token level. all the tokens do this independently
        return self.net(x)


In [45]:
# start to intersperse the communication with computation
# transformers does this when it has blocks that communicate and then computer, it groups them and replicates them
class Block(nn.Module):
    """ Transformer block: communication followed by computation"""

    def __init__(self, n_emb, n_head):
        # n_emb: emdedding dimension; n_head: the number of heads self-attention
        super().__init__()
        head_size = n_emb//n_head # we perform sel_attention to a equal part of the blocks
        self.sa = MultiHeadAttention(n_head, head_size) # performs computation between tokens
        self.ffwd = FeedForward(n_emb) # performs "thinking" on the tokens
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb) # make unit mean, unit gaussian at initializaion

    def __call__(self, x):
        x = x + self.sa(self.ln1(x)) # we fork off some communication and we come back using addition(+)
        
        x = x + self.ffwd(self.ln2(x)) # we fork off some communication and we come back using addition(+)
        # these two above with plus(+)/addition(+) forms residual connections 
        return x

In [46]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each token dirctly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb) # to create level of interaction. we dont't dont directly go to embeddings for logits
        self.posistion_embedding_table = nn.Embedding(block_size, n_emb) # each postion from 0 to block_size-1 will also get some embeddig vector 
        self.blocks = nn.Sequential(*[Block(n_emb, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_emb) # final layer norm
        self.lm_head = nn.Linear(n_emb, vocab_size) # use linear layer to go from token_embeddings to logits. this layer decodes into vocabulary
        
    def __call__(self, idx, targets=None):
        B,T = idx.shape
        
        # idx and targets are both targets of (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C). thic C -> n_emb
        pos_emb = self.posistion_embedding_table(mx.arange(T)) # this will be integers from 0 to T-1 and all these integers from 0 to T-1 get embedded through table to get (T,C), C -> n_emb. just like each token is mapped to the embedding table, each posistion is mapped using the posistion embedding table  
        x = tok_emb + pos_emb # (B,T,C). there will be broadcasting across batch. C -> n_emb. we add this postional embedding ie., posistions at which these tokens occur to every examples with the word embeddings. encoding information with token embeddings and posistional embeddings
        x = self.blocks(x)
        logits = self.lm_head(x) # (B,T,C). thic C -> vocab_size. output, 'x',from the previous step is fed into decoder language modelling head
        
        return logits

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:,-block_size:] # '-block_size:' as we are taking token from last
            # get predictions
            logits = self(idx_cond) # logits -> (B,T,C)
            # focus only on the last time step
            logits = logits[:,-1,:] # becomes (B, C). this line means that we are choosing the lateset generated character
            # apply softmax to get probabilities
            prob = mx.softmax(logits, axis=1) # (B, C)
            # sample from the distribution
            idx_next = random.categorical(prob, shape=(1,1)) # (B, 1)
            # append sampled index to the running sequence
            idx = mx.concatenate((idx, idx_next), axis=1) # (B, T+1). here for predicting we are inputting the entire prediction
        
        return idx

In [64]:
# so far we taken these indices, idx and encoded them based on the identity of tokens inside idx. 
# now we are not only encoding identity, but also their posistions
model = BigramLanguageModel()
logits = model(xb)

In [65]:
def generate(model, max_new_tokens):
    # we kick off generation with '0' ie., newline(\n)
    # starting with one batch and one character
    return decode(model.generate(idx = mx.zeros([1,1], dtype=mx.int64), max_new_tokens = max_new_tokens)[0].tolist())

In [66]:
# training the model
def loss_fn(model, X, y):
    logits = model(X)
    
    return nn.losses.cross_entropy(logits, y, reduction="mean")

loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

lr_schedule = optim.exponential_decay(init=1e-3, decay_rate=0.9)
optimizer = optim.Adam(learning_rate=lr_schedule)

mx.eval(model.parameters())

In [67]:
for steps in range(10):
    # sample batch of data
    xb, yb = get_batch('train')
    
    loss, grads = loss_and_grad_fn(model, xb, yb)

    optimizer.update(model, grads)
    mx.eval(loss, model.parameters())

    # print(f"Loss: {loss}")


In [68]:
print(f"Loss: {loss}")


Loss: 5.061156749725342


In [69]:
generate(model, 100)

"\nZXh3gyb?RinJqvoLLclHmZlJLxkXeQsxGW\ngR\nGD:kx'pE eZ,,ce:pRxoesAsyJ3Js!wZSICIB,eAOWxgHeOy O:K-nZ lj!wkD"